# Emotion Recognition in Tunisian Dialect Speech with wav2vec 2.0 XLSR

This notebook fine-tunes **wav2vec 2.0 XLSR** to classify the emotion of Tunisian-dialect speech into 4 classes: `angry`, `happy`, `neutral`, `sad`. It is the last stage of my end-of-study project, whose goal was a **speech emotion recognition (SER)** system to be integrated into voicebots.

**What happens in this notebook**
1. Load the train / test CSV files (audio path + emotion label)
2. Resample the audio to 16 kHz and turn it into model inputs
3. Build the model: pretrained wav2vec 2.0 encoder, then mean pooling, then a small classification head
4. Fine-tune with the Hugging Face `Trainer`
5. Evaluate on the test set (classification report)
6. Run inference on a single audio file

> **Note:** the audio corpus is not included in this repository. The outputs shown below were produced by the original training run on Google Colab (GPU). The code was tidied afterwards (see the README) and has not been re-run on the original data.

The model and training loop follow the Greek SER notebook of [m3hrdadfi/soxan](https://github.com/m3hrdadfi/soxan), adapted to Tunisian dialect.

## 1. Setup

In [ ]:
# torch and torchaudio are preinstalled on Google Colab
!pip install -q datasets transformers scikit-learn

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

All paths and key settings live in one place. **Edit this cell** to match your own Google Drive layout.

The CSV files are **tab-separated** and have three columns: `name` (file name), `path` (full path to the `.wav`) and `emotion` (label).

In [ ]:
from pathlib import Path

# --- Paths -----------------------------------------------------------------
DATA_DIR   = Path("/content/drive/MyDrive/tunisian-ser/data")  # holds train.csv and test.csv
TRAIN_CSV  = DATA_DIR / "train.csv"
TEST_CSV   = DATA_DIR / "test.csv"
OUTPUT_DIR = Path("./wav2vec2-xlsr-ser-checkpoints")   # Trainer checkpoints
MODEL_DIR  = Path("./wav2vec2-xlsr-ser-final")         # final model, saved at the end of training

# --- Model -----------------------------------------------------------------
BASE_MODEL   = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"  # XLSR-53 already fine-tuned on Arabic speech
POOLING_MODE = "mean"                                          # how frame features are merged: mean | sum | max
INPUT_COLUMN, OUTPUT_COLUMN = "path", "emotion"

## 2. Data

`datasets` reads the two CSV files. The test split is also used as the validation split during training.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files={"train": str(TRAIN_CSV), "validation": str(TEST_CSV)},
    delimiter="\t",
)
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['name', 'path', 'emotion'],
    num_rows: 5587
})
Dataset({
    features: ['name', 'path', 'emotion'],
    num_rows: 1025
})


The labels are sorted so that label to id mapping is deterministic.

In [ ]:
label_list = train_dataset.unique(OUTPUT_COLUMN)
label_list.sort()
num_labels = len(label_list)
print(f"A classification problem with {num_labels} classes: {label_list}")

A classification problem with 4 classes: ['angry', 'happy', 'neutral', 'sad']


## 3. Model configuration and audio preprocessing

The **processor** normalises the raw waveform and expects **16 kHz** audio. The model config receives the label mappings (`label2id` / `id2label`) so that they are saved with the model.

In [ ]:
from transformers import AutoConfig, Wav2Vec2Processor

config = AutoConfig.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    label2id={label: i for i, label in enumerate(label_list)},
    id2label={i: label for i, label in enumerate(label_list)},
    finetuning_task="wav2vec2_clf",
)
setattr(config, "pooling_mode", POOLING_MODE)

processor = Wav2Vec2Processor.from_pretrained(BASE_MODEL)
target_sampling_rate = processor.feature_extractor.sampling_rate
print(f"The target sampling rate: {target_sampling_rate}")

The target sampling rate: 16000


`load_audio` reads a wav file, mixes it down to mono if needed and resamples it to 16 kHz. `preprocess_function` converts a batch of rows into model inputs and integer labels.

In [ ]:
import torch
import torchaudio


def load_audio(path):
    """Read a wav file, mix down to mono and resample to the model's sampling rate."""
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != target_sampling_rate:
        waveform = torchaudio.transforms.Resample(sr, target_sampling_rate)(waveform)
    return waveform.squeeze().numpy()


def label_to_id(label, label_list):
    return label_list.index(label) if label in label_list else -1


def preprocess_function(examples):
    speech_list = [load_audio(path) for path in examples[INPUT_COLUMN]]
    target_list = [label_to_id(label, label_list) for label in examples[OUTPUT_COLUMN]]

    result = processor(speech_list, sampling_rate=target_sampling_rate)
    result["labels"] = list(target_list)
    return result

In [ ]:
train_dataset = train_dataset.map(preprocess_function, batch_size=100, batched=True, num_proc=4)
eval_dataset = eval_dataset.map(preprocess_function, batch_size=100, batched=True, num_proc=4)

## 4. Model

The architecture is a standard wav2vec 2.0 encoder followed by:
1. **pooling** over time (mean of all frame-level vectors) to get one vector per utterance,
2. a small **classification head**: dropout, dense layer, `tanh`, dropout, output layer with 4 units.

The loss is cross-entropy (single-label classification).

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple
import torch
from transformers.utils import ModelOutput


@dataclass
class SpeechClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None

In [ ]:
import torch
import torch.nn as nn
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss

from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2PreTrainedModel,
    Wav2Vec2Model
)


class Wav2Vec2ClassificationHead(nn.Module):
    """Head for wav2vec classification task."""

    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        x = features
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class Wav2Vec2ForSpeechClassification(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.pooling_mode = config.pooling_mode
        self.config = config

        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = Wav2Vec2ClassificationHead(config)

        self.init_weights()

    def freeze_feature_extractor(self):
        self.wav2vec2.feature_extractor._freeze_parameters()

    def merged_strategy(
            self,
            hidden_states,
            mode="mean"
    ):
        if mode == "mean":
            outputs = torch.mean(hidden_states, dim=1)
        elif mode == "sum":
            outputs = torch.sum(hidden_states, dim=1)
        elif mode == "max":
            outputs = torch.max(hidden_states, dim=1)[0]
        else:
            raise Exception(
                "The pooling method hasn't been defined! Your pooling mode must be one of these ['mean', 'sum', 'max']")

        return outputs

    def forward(
            self,
            input_values,
            attention_mask=None,
            output_attentions=None,
            output_hidden_states=None,
            return_dict=None,
            labels=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.wav2vec2(
            input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = outputs[0]
        hidden_states = self.merged_strategy(hidden_states, mode=self.pooling_mode)
        logits = self.classifier(hidden_states)

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SpeechClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

The **data collator** pads the waveforms of a batch to the same length (audio clips have different durations) and builds the label tensor.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Union

import torch
from transformers import Wav2Vec2Processor


@dataclass
class DataCollatorForSpeechClassification:
    """Dynamically pad the input values of a batch and attach the labels."""

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [f["labels"] for f in features]
        d_type = torch.long if isinstance(label_features[0], int) else torch.float

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor(label_features, dtype=d_type)
        return batch


data_collator = DataCollatorForSpeechClassification(processor=processor, padding=True)

The metric tracked during training is **accuracy**.

In [ ]:
import numpy as np
from transformers import EvalPrediction

is_regression = False


def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.squeeze(preds) if is_regression else np.argmax(preds, axis=1)

    if is_regression:
        return {"mse": ((preds - p.label_ids) ** 2).mean().item()}
    return {"accuracy": (preds == p.label_ids).astype(np.float32).mean().item()}

## 5. Fine-tuning

The pretrained weights are loaded and the classification head is initialised randomly (the warning about unused `lm_head` weights is expected: that head belonged to the original speech-recognition model).

The small convolutional **feature extractor** (the first layers that turn the waveform into frames) is **frozen**. The transformer layers and the new head are trained.

| Setting | Value |
|---|---|
| Epochs | 4 |
| Batch size | 4 per step, gradient accumulation 2 (effective batch 8) |
| Learning rate | 1e-4 |
| Evaluation / checkpoint | every 500 steps |
| Mixed precision (fp16) | off |

In [ ]:
model = Wav2Vec2ForSpeechClassification.from_pretrained(BASE_MODEL, config=config)
model.freeze_feature_extractor()

In [ ]:
import transformers
from packaging import version
from transformers import Trainer, TrainingArguments

# The argument was renamed from `evaluation_strategy` to `eval_strategy` in recent transformers versions.
eval_kw = "eval_strategy" if version.parse(transformers.__version__) >= version.parse("4.41") else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4.0,
    # fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=10,
    learning_rate=1e-4,
    save_total_limit=2,
    **{eval_kw: "steps"},
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [ ]:
trainer.train()

***** Running training *****
  Num examples = 5587
  Num Epochs = 4
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 2
  Total optimization steps = 2792

Step   Training Loss   Validation Loss   Accuracy
 500      0.812100        0.961475       0.611707
1000      0.558700        0.957534       0.658537
1500      0.463000        0.950506       0.688780
2000      0.434600        0.942607       0.695610
2500      0.267400        0.951094       0.697561

TrainOutput(global_step=2792, training_loss=0.6077971989368982,
            metrics={'train_runtime': 5809.2224, 'train_samples_per_second': 3.847,
                     'train_steps_per_second': 0.481, 'epoch': 4.0})


**Reading the log:** the training loss keeps going down (0.81 to 0.27) while the validation loss stays flat around 0.95. The validation accuracy still improves (61% to 70%) but this gap suggests the model starts to overfit the training set. See the README for ideas to improve this.

Save the model, its config and the processor, so that they can be reloaded for inference.

In [ ]:
model.save_pretrained(MODEL_DIR)
config.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)

## 6. Evaluation on the test set

The saved model is reloaded from disk (to check that saving worked) and used to predict the emotion of every test utterance.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR, local_files_only=True)
model = Wav2Vec2ForSpeechClassification.from_pretrained(MODEL_DIR, local_files_only=True).to(device).eval()

Device: cuda


In [ ]:
def predict_batch(batch):
    speech_list = [load_audio(path) for path in batch["path"]]
    features = processor(speech_list, sampling_rate=target_sampling_rate, return_tensors="pt", padding=True)

    input_values = features.input_values.to(device)
    attention_mask = features.attention_mask.to(device)

    with torch.no_grad():
        logits = model(input_values, attention_mask=attention_mask).logits

    batch["predicted"] = torch.argmax(logits, dim=-1).cpu().numpy()
    return batch


test_dataset = load_dataset("csv", data_files={"test": str(TEST_CSV)}, delimiter="\t")["test"]
result = test_dataset.map(predict_batch, batched=True, batch_size=8)

In [ ]:
from sklearn.metrics import classification_report

label_names = [config.id2label[i] for i in range(config.num_labels)]
y_true = [config.label2id[name] for name in result["emotion"]]
y_pred = result["predicted"]

print(classification_report(y_true, y_pred, target_names=label_names))

              precision    recall  f1-score   support

       angry       0.56      0.84      0.67       228
       happy       0.60      0.62      0.61       108
     neutral       0.79      0.54      0.64       394
         sad       0.78      0.81      0.79       295

    accuracy                           0.69      1025
   macro avg       0.69      0.70      0.68      1025
weighted avg       0.72      0.69      0.69      1025


**Summary:** accuracy **69%**, macro F1 **0.68**.

- `angry` has a high recall (0.84) but a low precision (0.56): the model predicts `angry` too often.
- `neutral` has the opposite profile (precision 0.79, recall 0.54): many neutral utterances are classified as something else.
- `sad` is the easiest class (F1 0.79). `happy` is the hardest (F1 0.61) and also the smallest class (108 test utterances).

## 7. Inference on a single audio file

`predict_emotion` returns the probability of each emotion for one wav file. This is the function to wrap in an API or a demo app.

In [ ]:
import torch.nn.functional as F
import pandas as pd


def predict_emotion(path):
    speech = load_audio(path)
    features = processor(speech, sampling_rate=target_sampling_rate, return_tensors="pt", padding=True)

    with torch.no_grad():
        logits = model(features.input_values.to(device), attention_mask=features.attention_mask.to(device)).logits

    scores = F.softmax(logits, dim=1).cpu().numpy()[0]
    return {config.id2label[i]: round(float(s) * 100, 1) for i, s in enumerate(scores)}


test_df = pd.read_csv(TEST_CSV, sep="\t")
for _, row in test_df.sample(5, random_state=0).iterrows():
    print(f"true: {row['emotion']:8s} -> predicted scores (%): {predict_emotion(row['path'])}")